# Lab 1: Forecasting Monthly Contribution Inflows

**Time:** about 75 minutes

The investment team asks you a straightforward question:

> *"How much contribution income will reach the Fund each month over the next year?"*

They need the answer to plan which investments the Fund can make, and when. You'll answer it with three kinds of
forecasting model, and one very simple benchmark that all three have to beat:

```
employer payments ─► monthly series ─► explore trend & seasonality ─► hold out the last 12 months
                  ─► naive baselines ─► SARIMA (statsmodels) ─► Prophet ─► compare ─► forecast with a range
```

### What is forecasting, and how is it different from Day 2?

On Day 2 each claim was a separate example, and the order of the claims didn't matter. A forecast is about a **time
series**: one number per month, in order, where each month is related to the months before it. Forecasting means
using the pattern of the past months to predict the coming months. Two ideas make that possible:

- **Trend:** the long-term direction. The Fund grows, so contributions tend to rise year after year.
- **Seasonality:** a pattern that repeats every year. If June is always high, next June will probably be high too.

A good forecast captures both, and also says **how sure it is**: "about 9 billion, most likely between 8.5 and 9.5"
is far more useful for planning than "9 billion".

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **time series** | a list of values in time order, e.g. total contributions for each month |
| **trend** | the slow, long-term direction of the series (rising, falling, flat) |
| **seasonality** | a pattern that repeats at the same time every year |
| **remainder** | what's left once trend and seasonality are taken out: the unexplained ups and downs |
| **stationary** | a series whose average and spread don't drift over time; classical models need this |
| **differencing** | working with the change from one month to the next instead of the values themselves |
| **horizon** | how far ahead you forecast; here 12 months |
| **baseline** | a very simple forecast that any real model has to beat |
| **hold-out / test period** | the last months, hidden from the models, used to check their forecasts |
| **MAE, RMSE, MAPE** | three ways of measuring how far a forecast was from what actually happened |
| **interval** | a range around the forecast; an 80% interval should contain the real value 8 times out of 10 |
| **SARIMA** | a classical forecasting model built from recent values, recent errors and the same month last year |
| **Prophet** | a forecasting library from Meta that fits a trend plus yearly patterns, and allows special events |

## Where this lab fits

**What you've done so far.** On Day 1 you cleaned the contributions table and noticed two things you'll need today:
the June spike in cash received (the year-end compliance drive) and the 2020 COVID-19 dip. On Days 2 and 3 you built
models that score one claim at a time, where the order of the rows didn't matter.

**What you'll do now.** You'll turn nine years of employer payments into one number per month, take the series apart
into trend, seasonal pattern and remainder, and forecast 2025 with three simple baselines and two real forecasting
models (SARIMA and Prophet). You'll find out, honestly, whether the models beat the baselines. Then you'll produce a
2026 forecast with a range for the investment team.

**Why it matters.** The Fund invests the contributions it receives. If the investment team expects more cash than
arrives, it may have to sell investments at a bad moment to pay benefits; if it expects less, money sits idle. A
forecast that's slightly better than the obvious guess, with an honest range, is worth a lot.

**What happens next.** Lab 2 forecasts the other side of the cash flow, benefit payouts, with a different tool: the
XGBoost you met on Day 2, re-used for time series. Lab 3 tests every forecasting method over several years instead of
one, and turns the results into a planning pack.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [ ]:
# ---- Setup for Day 4. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = []

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

## Imports

As well as pandas and matplotlib, this lab uses **statsmodels**, a Python library of classical statistics. Three tools
come from it:

- `STL` splits a series into trend, seasonal pattern and remainder (Section 2);
- `adfuller` runs a test that tells you whether a series is stationary (Section 2);
- `SARIMAX` is the SARIMA forecasting model (Section 5).

Prophet is imported later, when it's needed. The `logging` lines switch off Prophet's very long progress messages.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# statsmodels: classical statistics and time-series models.
from statsmodels.tsa.seasonal import STL                       # split a series into trend + season + rest
from statsmodels.tsa.statespace.sarimax import SARIMAX         # the SARIMA model
from statsmodels.tsa.stattools import adfuller                 # the stationarity test

# Hide library notices, and Prophet's very chatty progress messages.
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("prophet.plot").disabled = True     # "plotly not installed": we only need static charts

A few settings used throughout: the chart size, four colours, and the **horizon**: how many months ahead to forecast.

In [ ]:
plt.rcParams["figure.figsize"] = (11, 4)     # default chart size (width, height in inches)

# Colours used in the charts.
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GREEN = "#1baf7a"
GREY = "#8a8984"

# Forecast 12 months ahead.
HORIZON = 12

> **Where we are.** The libraries are loaded, and the horizon is set to 12 months. First you need a series to
> forecast: the contributions table has thousands of employer rows per month, and a forecast needs one number per
> month.

## 1. From employer payments to a monthly series

The data file has one row per employer per wage month: what the employer owed (`contribution_due_tzs`), what it
actually paid (`amount_paid_tzs`), and when it paid (`payment_date`).

A forecasting model doesn't want thousands of employer rows. It wants **one number per month**. There are two ways to
turn the file into a monthly series, and they answer different questions:

- **By wage month** (`period`): how much employers *owe* for each month's wages. This is the "due" series.
- **By payment month** (`payment_date`): how much cash actually *arrived* each month. This is the "cash" series.

They're not the same: an employer that pays March's contributions in May adds to March's "due" but to May's "cash".

In [ ]:
contrib = pd.read_csv("contributions_monthly_clean.csv", parse_dates=["due_date", "payment_date"])
contrib.head()

**The "due" series.** Turn the `period` text (`"2024-03"`) into a real date (1 March 2024), then add up the amount due
for each month across all employers. Dividing by `1e9` (one billion) gives TZS billions, which are easier to read.

In [ ]:
# Series 1: contributions DUE, by wage month.
# "2024-03" -> 2024-03-01, the first day of the month.
contrib["wage_month"] = pd.to_datetime(contrib["period"] + "-01")

# Add up the amount due for each month, across all employers.
due = contrib.groupby("wage_month")["contribution_due_tzs"].sum() / 1e9    # in TZS billions
due.head()

**The "cash" series.** Keep only rows that were actually paid, find the month each payment arrived in, and add up the
amounts paid per month. `dt.to_period("M").dt.to_timestamp()` turns any date into the first day of its month
(17 April 2024 → 1 April 2024), so all payments in the same month are grouped together.

In [ ]:
# Series 2: CASH received, by the month it was paid.
paid = contrib[contrib["payment_date"].notna()].copy()     # only rows with a payment

# The first day of the payment month: 2024-04-17 -> 2024-04-01.
paid["payment_month"] = paid["payment_date"].dt.to_period("M").dt.to_timestamp()

cash = paid.groupby("payment_month")["amount_paid_tzs"].sum() / 1e9
cash.head()

Draw both series on one chart to see how they differ.

In [ ]:
plt.plot(due.index, due.values, color=GREY, label="due, by wage month")
plt.plot(cash.index, cash.values, color=BLUE, label="cash received, by payment month")
plt.ylabel("TZS billions")
plt.title("Two views of the same contributions", loc="left")
plt.legend()
plt.show()

Look at the two lines. The grey one is smooth. The blue one has spikes every June, which is the year-end
compliance drive pulling in arrears, and a dip in 2020 when employers struggled to pay during COVID-19.

The investment team needs to know when **cash** arrives, so that's what you'll forecast. Before you do,
trim the edges. The first month has only a handful of payments (the extract starts in January 2017 and those
contributions weren't due until February), and the last month in the file is January 2026, which is the
month the extract was taken, so it's incomplete.

A forecasting model also needs the series to be **continuous**: exactly one value for every month, with no missing
months. `asfreq("MS")` ("month start") makes pandas check that: it would insert an empty row for any month that's
missing, so counting empty values tells you whether there are gaps.

In [ ]:
# Keep March 2017 to December 2025.
y = cash.loc["2017-03-01":"2025-12-01"]

# asfreq("MS") says "one value per month, on the month start". It would add an empty row for any missing month.
y = y.asfreq("MS")

print(len(y), "months, from", y.index.min().date(), "to", y.index.max().date())
print("missing months:", y.isna().sum())

106 months and no gaps. `y` is the series you'll forecast.

> **Where we are.** You have the cash series `y`: 106 months, March 2017 to December 2025, with no gaps. Before
> choosing a model, you'll take it apart to see what it's made of.

## 2. Trend, seasonality and noise

Before choosing a model, understand the series. **STL** (Seasonal-Trend decomposition using LOESS) pulls a series apart
into three pieces that add back up to the original:

- the **trend**: the slow-moving long-term level;
- the **seasonal** part: the pattern that repeats every 12 months;
- the **remainder**: whatever is left over, the irregular ups and downs neither of the others explains.

It's done on the **log** of the series. Why? The Fund grows, so a "June is 25% higher" pattern means a bigger jump in
shillings every year. On a log scale, "25% higher" is always the same distance, so the seasonal pattern becomes the
same size every year, which is what STL assumes.

In [ ]:
log_y = np.log(y)

# period=12: the pattern repeats every 12 months. robust=True: don't let a few odd months (2020) distort it.
stl = STL(log_y, period=12, robust=True).fit()

Draw the original series and its three pieces, one above the other. They share the same time axis.

In [ ]:
# Four charts stacked: the series, then its three parts.
fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
axes[0].plot(log_y, color=BLUE)
axes[0].set_ylabel("observed (log)")
axes[1].plot(stl.trend, color=BLUE)
axes[1].set_ylabel("trend")
axes[2].plot(stl.seasonal, color=BLUE)
axes[2].set_ylabel("seasonal")
axes[3].plot(stl.resid, color=BLUE)
axes[3].set_ylabel("remainder")
plt.show()

The seasonal part is in logs, which is hard to read. `exp(s) - 1` turns it back into a percentage: how far above or
below the trend each month typically is. Average it over the years for each calendar month (1 = January … 12 = December).

In [ ]:
# The seasonal part is in logs. exp(s) - 1 turns it into "% above or below the trend".
seasonal_pct = np.exp(stl.seasonal) - 1

# Average for each calendar month (1 = January ... 12 = December), shown as a percentage.
season = seasonal_pct.groupby(seasonal_pct.index.month).mean()
(season * 100).round(1)

In [ ]:
season.plot(kind="bar", color=BLUE, figsize=(8, 3))
plt.title("Average seasonal effect by calendar month (0.10 = 10% above trend)", loc="left")
plt.xlabel("month")
plt.ylabel("vs trend")
plt.show()

June stands out clearly. The other months are within a few percent of the trend. The remainder shows the
2020 shock: a period the seasonal pattern can't explain.

### Stationarity: can a classical model use this series as it is?

Classical models such as ARIMA (AutoRegressive Integrated Moving Average) assume the series is **stationary**: its
average level and its spread stay roughly the same over time. Think of a series that wobbles around a flat line. Our
series clearly isn't like that: it climbs steadily.

The usual fix is **differencing**: instead of modelling the amount each month, model the **change** from the month
before. A steadily rising series becomes a series of roughly constant changes, which *is* stationary. A **seasonal
difference** (this June minus last June) removes the yearly pattern in the same way.

The **Augmented Dickey-Fuller (ADF) test** checks stationarity and gives a **p-value**. Read it like this: a small
p-value (below 0.05) is evidence that the series *is* stationary; a large one means it probably isn't. Test the log
series, then the differenced versions.

In [ ]:
# adfuller returns several numbers; the second one ([1]) is the p-value.
p_log = adfuller(log_y)[1]
print(f"log series:                        p = {p_log:.3f}")

# diff() = this month minus last month. The first value becomes empty, so dropna().
first_difference = log_y.diff().dropna()
p_diff = adfuller(first_difference)[1]
print(f"after first difference:            p = {p_diff:.3f}")

# diff(12) = this month minus the same month last year.
seasonal_difference = log_y.diff().diff(12).dropna()
p_seasonal = adfuller(seasonal_difference)[1]
print(f"after first + seasonal difference: p = {p_seasonal:.3f}")

The raw log series has a large p-value (not stationary); after differencing it's tiny (stationary). Differencing
(working with month-to-month changes rather than levels) removes the trend. A seasonal difference
(this June minus last June) removes the yearly pattern. That's what the `d=1` and `D=1` in the SARIMA model below
will do for you.

> **Where we are.** You know the series has a rising trend, a strong June peak, and a COVID shock in 2020, and that
> it needs differencing before a classical model can use it. Before fitting anything, set aside the data you'll test
> on.

## 3. Hold out the last year

To judge a forecast you have to compare it with what really happened. So **hide the last 12 months (2025)** from every
model: train on everything up to December 2024, forecast 2025, then compare the forecast with the real 2025 values.

This is the forecasting version of Day 2's split by date: the models are tested on a period they've never seen, just as
they'll be used on a future they can't see.

In [ ]:
# All but the last 12 months for training; the last 12 for testing.
train = y[:-HORIZON]
test = y[-HORIZON:]

print("train:", train.index.min().date(), "to", train.index.max().date(), f"({len(train)} months)")
print("test: ", test.index.min().date(), "to", test.index.max().date(), f"({len(test)} months)")

### How to measure a forecast's error

For each month, the **error** is the actual value minus the forecast. Three standard ways to turn 12 monthly errors into
one number:

- **MAE** (mean absolute error): the average size of the error, ignoring whether it was too high or too low, in TZS
  billions. Easiest to explain: "on average we were 0.2 billion off".
- **RMSE** (root mean squared error): similar, but squares the errors first, so one big miss counts much more than
  several small ones. Useful when a big miss is especially costly.
- **MAPE** (mean absolute percentage error): the average error as a percentage of the actual value. Good for comparing
  series of different sizes.

For all three, **smaller is better**. You'll measure every forecast this way, so write it once as a function:

In [ ]:
def metrics(actual, forecast):
    """MAE, RMSE and MAPE of a forecast, as a dictionary."""
    error = actual - forecast                              # one error per month
    mae = error.abs().mean()                               # average size of the error
    rmse = np.sqrt((error ** 2).mean())                    # square, average, square root
    mape = (error.abs() / actual).mean() * 100             # average error as % of the actual value
    # float() and round() just make the numbers print neatly.
    return {"MAE": round(float(mae), 3), "RMSE": round(float(rmse), 3), "MAPE %": round(float(mape), 2)}

> **Where we are.** 2017-2024 is for training and 2025 is hidden for testing, and you have one function that scores
> any forecast with MAE, RMSE and MAPE. Now for the forecasts themselves, starting with the simplest.

### Theory: why simple forecasts are hard to beat

It seems obvious that a sophisticated model should beat "the same month last year". In practice, for business series
like this one, it often doesn't. Three reasons:

- **Most of the pattern is simple.** If a series is mainly "steady growth plus a June peak", the seasonal naive with
  growth already captures nearly all of it. There's little left for a clever model to find.
- **Clever models can learn noise.** With only eight years of monthly data (about 96 points), a model with many
  settings can fit the past's random wobbles and carry them into the future.
- **The future isn't the past.** Every model assumes the patterns it learned will continue. When something changes (a
  new rule, an economic shock), the simple forecast and the clever one are both wrong.

That's why forecasting teams have a firm rule: report every model's error **next to a baseline's**. A number like
"3.9% error" means nothing on its own.

## 4. Baselines: the forecasts any model must beat

Before building anything sophisticated, make the simplest forecasts possible. If a clever model can't beat them, it
isn't worth the extra effort, cost and explaining. Three baselines:

- **Naive:** every month of 2025 equals the last month you saw (December 2024).
- **Seasonal naive:** each month of 2025 equals the same month of 2024 (January = last January, and so on). This
  captures the June spike automatically.
- **Seasonal naive with growth:** the same month last year, scaled up by last year's growth rate. This captures both
  the seasonal pattern and the trend.

Each takes one or two lines. Keep every forecast in a dictionary (name → monthly values) so they can all be scored
together later.

In [ ]:
# Keep every forecast in one dictionary: name -> monthly values for 2025.
forecasts = {}

# Naive: repeat the last value 12 times.
last_value = train.iloc[-1]
forecasts["Naive"] = pd.Series(last_value, index=test.index)

# Seasonal naive: 2025's months = 2024's months.
last_12_months = train.iloc[-12:].values
forecasts["Seasonal naive"] = pd.Series(last_12_months, index=test.index)

For the growth version, work out last year's growth: the total of the last 12 months of training data divided by the
total of the 12 months before that. A result of 1.094 means "9.4% higher".

In [ ]:
# Growth: the total of the last 12 months divided by the total of the 12 months before.
total_last_year = train.iloc[-12:].sum()
total_year_before = train.iloc[-24:-12].sum()
growth = total_last_year / total_year_before
print(f"growth used: {growth - 1:+.1%}")

# Same month last year, times the growth.
forecasts["Seasonal naive + growth"] = forecasts["Seasonal naive"] * growth

Score the three baselines against the real 2025 values:

In [ ]:
print("Naive:                  ", metrics(test, forecasts["Naive"]))
print("Seasonal naive:         ", metrics(test, forecasts["Seasonal naive"]))
print("Seasonal naive + growth:", metrics(test, forecasts["Seasonal naive + growth"]))

> **Where we are.** The three baselines are scored, and the best of them sets the bar. Now the first real model.

### Theory: autocorrelation, the idea SARIMA is built on

Forecasting from a series' own past works because neighbouring months are related. **Autocorrelation** measures how
strongly a series is correlated with itself shifted back in time:

- **Lag 1:** is this month like last month? For contributions, very much so: if March was high, April probably is too.
- **Lag 12:** is this month like the same month last year? For a seasonal series, yes: June resembles last June.

SARIMA turns those relationships into a formula. In words, its forecast for next month is roughly:

```
next month's change = a × (last month's change)            ← AR: past values
                    + b × (last month's forecast error)     ← MA: past mistakes
                    + c × (same month last year's error)    ← seasonal MA
```

It fits the numbers `a`, `b`, `c` to the training data, and because it knows how big its errors typically were, it can
also say how uncertain each forecast is. The further ahead you forecast, the more those uncertainties pile up, so the
interval widens month by month.

## 5. SARIMA

**SARIMA** stands for Seasonal AutoRegressive Integrated Moving Average. It's the classic statistical forecasting model,
and it predicts each month from three ingredients:

- **AR (autoregressive):** the recent values themselves ("last month was high, so this month probably is too");
- **MA (moving average):** the recent *errors* ("we under-forecast last month, so adjust");
- **S (seasonal):** the same ingredients, but from the same month last year.

The **I** (integrated) is the differencing from Section 2, done for you inside the model.

You choose how many of each ingredient to use with two sets of numbers, `order=(p, d, q)` and
`seasonal_order=(P, D, Q, 12)`:

- `p` / `P`: how many past values (months / years) to use;
- `d` / `D`: how many times to difference (month-to-month / year-to-year); 1 removes a trend;
- `q` / `Q`: how many past errors to use;
- `12`: the season length, 12 months.

`(1,1,1)(0,1,1,12)` is a sensible starting point for monthly business data. It's so common it has a nickname, the
"airline model", after the airline-passenger dataset it was first used on.

### Why model the logarithm, then convert back?

The Fund's inflows grow over time, and the size of the June spike grows with them. On a log scale, "grows by 8% a year"
becomes a straight line, and a seasonal swing of "+30% in June" becomes the same size every year. Both are much easier
for a model to learn. The forecast comes out in logs, and `np.exp` turns it back into TZS billions.

In [ ]:
# Fit SARIMA on the log of the training series.
sarima = SARIMAX(np.log(train), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
sarima = sarima.fit(disp=False)        # disp=False: don't print progress

# The fitted coefficients: the weight given to each ingredient. P>|z| below 0.05 means the ingredient clearly helps.
print(sarima.summary().tables[1])

### Theory: what a forecast interval means

A forecast is the model's best single guess. An **interval** is a range that should contain the real value a stated
share of the time. "An 80% interval of 8.5 to 9.5 billion" means: if you made many forecasts like this one, about 8 in
10 real values would land inside their ranges, and about 2 in 10 outside.

Two points beginners often miss:

- **Wider isn't worse.** An honest wide range is more useful than a confident narrow one that turns out wrong. Planners
  can prepare for a range; they can't prepare for a surprise.
- **The interval can be checked.** Once 2025 is over, count how many months fell inside. If an 80% interval caught only
  half of them, the model was overconfident. You'll do that check in Section 7.

Now forecast the 12 months of 2025. Besides the forecast itself, SARIMA gives an **interval**: a range it expects the real
value to fall in. `alpha=0.2` asks for an **80% interval** (20% chance of falling outside), which is a common choice
for planning: wide enough to be honest, narrow enough to be useful.

Both the forecast and the interval come out in logs, so `np.exp` converts them back to TZS billions.

In [ ]:
# Forecast 12 months ahead.
sarima_forecast = sarima.get_forecast(HORIZON)

# The forecast is in logs: exp() turns it back into TZS billions.
forecasts["SARIMA"] = np.exp(sarima_forecast.predicted_mean)

# An 80% interval (alpha=0.2 means 20% outside), also converted back from logs.
sarima_ci = np.exp(sarima_forecast.conf_int(alpha=0.2))
sarima_ci.columns = ["lower", "upper"]
sarima_ci.head()

Because the model was fitted on logs, the forecast and its interval are converted back with `exp`. The
interval is asymmetric in shillings: there's more room above the forecast than below it, which is typical for
things that grow.

> **Where we are.** SARIMA has produced a 2025 forecast with an 80% interval. Prophet takes a different approach to the
> same problem, one built from parts you can see and explain.

### Theory: forecasting by adding up parts

Prophet treats a series as a sum of separate, readable pieces (on the log scale, so the pieces are percentages):

```
log(value) = trend + yearly pattern + effect of special events + noise
```

- The **trend** is a line that can change slope at a few points.
- The **yearly pattern** is a smooth curve that repeats every 12 months.
- Each **special event** (here, COVID) gets its own column and its own learned effect.

This is a different philosophy from SARIMA. SARIMA asks "how does this month follow from the last few?". Prophet asks
"what are the building blocks of the series?". Prophet's parts are easier to explain to a committee; SARIMA often reacts
faster to recent changes. Neither is always better, which is why you test both.

## 6. Prophet

**Prophet** is a forecasting library from Meta (Facebook). It builds the forecast from parts you can see and explain:

- a **trend** that's allowed to bend at a few points ("changepoints") when growth speeds up or slows down;
- a **yearly pattern**, learned from the data (here, the June peak);
- optional **extra factors** you tell it about, called *regressors*: special events that explain unusual months.

That last part matters here. The 2020 COVID-19 disruption isn't something that should repeat every year. If Prophet
doesn't know about it, it will try to explain the 2020 dip with the trend or the seasonal pattern, and carry that damage
into the forecast. So you'll give Prophet a `covid` column: 1 for April to September 2020, 0 for every other month.

Prophet expects its data in a table with exactly two columns: **`ds`** (the date) and **`y`** (the value). As with
SARIMA, the log of the series is used.

In [ ]:
from prophet import Prophet

# Prophet wants a table with two columns: "ds" (the date) and "y" (the value). We use the log again.
df_train = pd.DataFrame({"ds": train.index, "y": np.log(train.values)})

# The extra column: 1 for April-September 2020, 0 for every other month.
is_covid = (df_train["ds"] >= "2020-04-01") & (df_train["ds"] <= "2020-09-30")
df_train["covid"] = is_covid.astype(int)

# Show the six months flagged as COVID.
df_train[df_train["covid"] == 1]

Create the model and fit it. The settings:

- `yearly_seasonality=True`: learn the yearly pattern;
- `weekly_seasonality=False`, `daily_seasonality=False`: switch these off, because the data is monthly;
- `interval_width=0.8`: give an 80% interval, the same as SARIMA;
- `changepoint_prior_scale=0.05`: how freely the trend may bend; small values keep it steady.

`add_regressor("covid")` tells Prophet to use the extra column.

In [ ]:
# Monthly data, so switch off weekly and daily patterns. interval_width=0.8 gives an 80% interval.
prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                  interval_width=0.8, changepoint_prior_scale=0.05)

# Tell Prophet about the covid column, then fit.
prophet.add_regressor("covid")
prophet.fit(df_train)
print("Prophet fitted")

To forecast, give Prophet a table of the dates to predict, with the same extra column. `covid = 0` for 2025: we don't
expect a repeat. The prediction has a `yhat` column (the forecast) and `yhat_lower` / `yhat_upper` (the interval), all
in logs, so `np.exp` converts them back.

In [ ]:
# The months to forecast. covid = 0: we don't expect a repeat.
future = pd.DataFrame({"ds": test.index})
future["covid"] = 0

prediction = prophet.predict(future)

# yhat is the forecast; yhat_lower and yhat_upper the interval. All in logs, so exp() them.
forecasts["Prophet"] = pd.Series(np.exp(prediction["yhat"].values), index=test.index)
prophet_ci = pd.DataFrame({
    "lower": np.exp(prediction["yhat_lower"].values),
    "upper": np.exp(prediction["yhat_upper"].values),
}, index=test.index)
prophet_ci.head()

One of Prophet's strengths is that you can see **what it learned**, part by part. The components chart shows the trend,
the yearly pattern, and the effect of the extra (covid) column.

In [ ]:
# What Prophet learned, part by part.
fig = prophet.plot_components(prophet.predict(df_train))
plt.show()

The components plot shows what Prophet learned: a steadily rising trend, a yearly pattern with its peak in June,
and the size of the COVID effect.

> **Where we are.** You have five forecasts for 2025: three baselines, SARIMA and Prophet, each from training data up to
> December 2024. Time to see which was closest to what really happened.

## 7. The comparison

Now score every forecast (three baselines and two models) the same way, against the real 2025 values, and sort with the
best (lowest MAE) at the top.

In [ ]:
# One row per forecast, best (lowest MAE) first.
rows = {}
for name in forecasts:
    rows[name] = metrics(test, forecasts[name])

results = pd.DataFrame(rows).T.sort_values("MAE")
results.round(3)

A chart helps you see *where* each forecast went wrong. Plot the real values from 2022 onwards (black), three of the
forecasts for 2025, and Prophet's 80% interval as a shaded band. The vertical line marks where the forecasts begin.

In [ ]:
# The actual values since 2022 against three of the forecasts.
actual_recent = y.loc["2022":]
plt.plot(actual_recent.index, actual_recent.values, color="black", lw=1.5, label="actual")

plt.plot(test.index, forecasts["Seasonal naive + growth"], color=GREY, ls="--", label="Seasonal naive + growth")
plt.plot(test.index, forecasts["SARIMA"], color=ORANGE, label="SARIMA")
plt.plot(test.index, forecasts["Prophet"], color=GREEN, label="Prophet")

# Shade Prophet's 80% interval.
plt.fill_between(test.index, prophet_ci["lower"], prophet_ci["upper"], color=GREEN, alpha=0.15, label="Prophet 80% interval")
plt.axvline(test.index[0], color=GREY, lw=0.8)

plt.ylabel("TZS billions")
plt.title("2025 forecasts against what actually arrived", loc="left")
plt.legend(ncol=3)
plt.show()

Now answer the question this lab is really about: **does Prophet (or SARIMA) actually outperform the baseline?**

Look at the MAE (mean absolute error) column, and be honest about it: on 2025, **the seasonal naive with growth wins.** Neither
SARIMA nor Prophet beats a forecast you could make in a spreadsheet in two minutes.

That isn't a failure of the lab. It's the most common outcome in real forecasting work, and the reason the
baseline rule exists. This series is dominated by a steady trend and one strong seasonal month, and "same month
last year, plus last year's growth" captures exactly that. A team that skipped the baseline would have deployed
Prophet and reported a 3.9% error as a success, without knowing that the spreadsheet did better.

Before you throw the models away, remember that one year is a small test. Lab 3 tests all of them over several
years. And the models still give you two things the baseline doesn't:

- **An interval.** "Between 8.5 and 9.5 billion in June, most likely about 9" is far more useful to an
  investment committee than "9".
- **Explanations.** Prophet's components let you say *why* June is high.

### Is the interval honest?

An 80% interval should contain the real value in roughly 8 months out of 10. If it contains it in only 5 out of 10, the
model is **overconfident** (its range is too narrow). If it contains it every single month, the range may be too wide to
be useful. Check both models:

In [ ]:
# True for each month where the actual value fell inside the interval. The mean of True/False = the share.
inside_prophet = (test >= prophet_ci["lower"]) & (test <= prophet_ci["upper"])
inside_sarima = (test >= sarima_ci["lower"]) & (test <= sarima_ci["upper"])

print(f"Prophet 80% interval contained the actual value in {inside_prophet.mean():.0%} of 2025 months")
print(f"SARIMA  80% interval contained the actual value in {inside_sarima.mean():.0%} of 2025 months")

> **Where we are.** On 2025, the seasonal naive with growth was the most accurate, and you've checked whether each
> model's 80% interval was honest. The models still bring something the baseline can't: a range and an explanation.
> Now produce the forecast the investment team will actually use.

### Theory: why you refit before forecasting for real

During testing, 2025 was hidden so you could check the forecasts. Once testing is over, keeping it hidden would throw
away the most recent and most relevant year. So the final model is **refitted on all the data**, including 2025, and
then forecasts 2026. The test told you *which method* to trust and *how accurate* it's likely to be; the refit gives that
method the newest information.

## 8. Forecast the next 12 months

The test is over. To produce the forecast you'd actually hand to the investment team, **refit on all the data,
including 2025**, so the model uses the most recent information, and forecast 2026.

In [ ]:
# All months, with the covid column.
df_all = pd.DataFrame({"ds": y.index, "y": np.log(y.values)})
is_covid = (df_all["ds"] >= "2020-04-01") & (df_all["ds"] <= "2020-09-30")
df_all["covid"] = is_covid.astype(int)

# The same Prophet settings as before, fitted on everything.
final = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                interval_width=0.8, changepoint_prior_scale=0.05)
final.add_regressor("covid")
final.fit(df_all)
print("Prophet fitted on all months")

Forecast the 12 months of 2026 and turn the result into a readable table for the investment team: one row per month,
with the forecast and the low and high ends of the 80% interval, in TZS billions.

In [ ]:
# The 12 months of 2026. date_range makes 12 month-start dates from January 2026.
future_2026 = pd.DataFrame({"ds": pd.date_range("2026-01-01", periods=HORIZON, freq="MS")})
future_2026["covid"] = 0
f26 = final.predict(future_2026)

# The plan for the investment team, back in TZS billions.
plan = pd.DataFrame({
    "forecast": np.exp(f26["yhat"]),
    "low_80": np.exp(f26["yhat_lower"]),
    "high_80": np.exp(f26["yhat_upper"]),
})
plan.index = future_2026["ds"].dt.strftime("%b %Y")       # "Jan 2026", "Feb 2026", ...
plan.round(2)

The yearly total is the sum of the monthly forecasts. It's tempting to add up the monthly low and high values too, to
get a range for the year:

In [ ]:
print(f"2026 total: {plan['forecast'].sum():.1f} bn TZS")
print(f"(sum of monthly 80% bounds: {plan['low_80'].sum():.1f} to {plan['high_80'].sum():.1f})")

Be careful with that last line. Adding up monthly lower bounds gives a range that's **too wide** for the annual
total, because months aren't all likely to be low at the same time. Lab 3 shows a better way to put a range on
an annual figure.

---

## Looking back, looking ahead

**What you did in this lab.** You turned employer payments into a monthly cash series, decomposed it into trend, seasonal
pattern and remainder, tested it for stationarity, held out 2025, built three baselines, fitted SARIMA and Prophet,
compared all five honestly, checked their intervals, and produced a 2026 forecast with a range.

**The ideas to carry forward.**

- Know which series you're forecasting: money owed (due) or cash received.
- Always compare with a naive baseline. Here the baseline won, and that's a real result.
- Give a range, not just a line, and check the range was honest.
- Test, then refit on all the data before forecasting for real.
- Summing monthly bounds gives a range for the year that's far too wide (Lab 3 fixes that).

**What happens next.** Lab 2 forecasts benefit payouts, using a machine learning model (XGBoost) instead of a
time-series model. To do that you'll learn how to turn a time series into an ordinary table of features.

## Your turn

1. **Due instead of cash.** Repeat the seasonal-naive-with-growth and SARIMA comparison on the `due` series
   (contributions owed by wage month). Is it easier or harder to forecast than cash? Why?
2. **Other SARIMA orders.** Try `(0,1,1)(0,1,1,12)` and `(2,1,0)(1,1,0,12)`. Compare their AIC (Akaike information criterion)
   (`model.aic`, lower is better) and their test MAE. Does the lowest AIC give the lowest test error?
3. **Forget COVID.** Refit Prophet without the `covid` regressor. What happens to the 2025 forecast and to the
   trend in the components plot?

In [ ]:
# Your code here

## Suggested answers

**(1)** The "due" series, trimmed and split exactly like the cash series, then the same two forecasts:

In [ ]:
# The due series, trimmed and split the same way.
y_due = due.loc["2017-03-01":"2025-12-01"].asfreq("MS")
train_due = y_due[:-HORIZON]
test_due = y_due[-HORIZON:]

# Seasonal naive + growth.
growth_due = train_due.iloc[-12:].sum() / train_due.iloc[-24:-12].sum()
snaive_due = pd.Series(train_due.iloc[-12:].values * growth_due, index=test_due.index)

# SARIMA, on logs, converted back with exp.
sarima_due = SARIMAX(np.log(train_due), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
sarima_due_forecast = np.exp(sarima_due.forecast(HORIZON))

print("Seasonal naive + growth:", metrics(test_due, snaive_due))
print("SARIMA:                 ", metrics(test_due, sarima_due_forecast))

The *due* series is much smoother: it depends on headcount and wages, which move slowly, so the baseline's
error drops to well under 1%. The *cash* series adds
payment behaviour on top (late payers, the June drive, COVID), and that's the hard part to forecast. It's also
the part finance cares about most. A useful approach in practice: forecast *due* accurately, then model the
*collection pattern* (what share of each month's dues arrives 0, 1, 2... months later) separately.

**(2)** Three SARIMA versions. **AIC** (Akaike information criterion) scores how well a model fits its *training* data,
with a penalty for using more ingredients; lower is better. It's a quick guide for choosing between versions, but the
real test is the error on the hidden year.

In [ ]:
# Model A: the one you used.
model_a = SARIMAX(np.log(train), order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
mae_a = metrics(test, np.exp(model_a.forecast(HORIZON)))["MAE"]
print(f"(1,1,1)(0,1,1,12)  AIC {model_a.aic:7.1f}   test MAE {mae_a:.3f}")

# Model B: no AR ingredient.
model_b = SARIMAX(np.log(train), order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
mae_b = metrics(test, np.exp(model_b.forecast(HORIZON)))["MAE"]
print(f"(0,1,1)(0,1,1,12)  AIC {model_b.aic:7.1f}   test MAE {mae_b:.3f}")

# Model C: two AR ingredients, a seasonal AR, no MA.
model_c = SARIMAX(np.log(train), order=(2, 1, 0), seasonal_order=(1, 1, 0, 12)).fit(disp=False)
mae_c = metrics(test, np.exp(model_c.forecast(HORIZON)))["MAE"]
print(f"(2,1,0)(1,1,0,12)  AIC {model_c.aic:7.1f}   test MAE {mae_c:.3f}")

AIC measures fit on the training data, with a penalty for complexity. It's a guide, not a guarantee. The order
with the best AIC doesn't always win on the test year. That's another reason for the backtests in Lab 3.

**(3)** The same Prophet model without the covid column:

In [ ]:
# The same Prophet, without the covid column.
no_covid = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                   changepoint_prior_scale=0.05)
no_covid.fit(df_train[["ds", "y"]])
print("fitted without covid")

no_covid_prediction = no_covid.predict(pd.DataFrame({"ds": test.index}))
forecast_no_covid = pd.Series(np.exp(no_covid_prediction["yhat"].values), index=test.index)

print("with covid flag:", metrics(test, forecasts["Prophet"]))
print("without:        ", metrics(test, forecast_no_covid))